In [1]:
"""
Dataset Analysis Script
Outputs all findings to dataset_analysis.txt
Run: conda activate miniproject && python analyze_dataset.py
"""

import os
import sys
from pathlib import Path
from PIL import Image
import numpy as np
from collections import defaultdict

OUTPUT_FILE = "dataset_analysis.txt"

def log(msg="", file=None):
    print(msg)
    if file:
        file.write(msg + "\n")

def analyze_split(split_path: Path, split_name: str, file):
    log(f"\n{'='*50}", file)
    log(f"  {split_name.upper()}  →  {split_path}", file)
    log(f"{'='*50}", file)

    if not split_path.exists():
        log(f"  !! PATH DOES NOT EXIST: {split_path}", file)
        return

    class_dirs = sorted([d for d in split_path.iterdir() if d.is_dir()])
    if not class_dirs:
        log("  !! No class subdirectories found.", file)
        return

    total = 0
    class_counts = {}
    size_info    = defaultdict(list)   # class -> list of (w, h)
    mode_info    = defaultdict(set)    # class -> set of PIL modes
    corrupt      = []

    for cls_dir in class_dirs:
        imgs = (list(cls_dir.glob("*.jpg"))  +
                list(cls_dir.glob("*.jpeg")) +
                list(cls_dir.glob("*.png"))  +
                list(cls_dir.glob("*.bmp")))
        class_counts[cls_dir.name] = len(imgs)
        total += len(imgs)

        # Sample up to 50 images for size/mode check
        for p in imgs[:50]:
            try:
                img = Image.open(p)
                size_info[cls_dir.name].append(img.size)   # (w, h)
                mode_info[cls_dir.name].add(img.mode)
                img.close()
            except Exception as e:
                corrupt.append((str(p), str(e)))

    # --- counts ---
    log(f"\n  Total images : {total}", file)
    log(f"  Num classes  : {len(class_counts)}", file)
    log(f"\n  Per-class counts:", file)
    max_count = max(class_counts.values()) if class_counts else 1
    for cls, cnt in class_counts.items():
        bar   = "█" * int(30 * cnt / max_count)
        pct   = cnt / total * 100 if total else 0
        log(f"    {cls:<30} {cnt:>5}  ({pct:5.1f}%)  {bar}", file)

    # --- imbalance ratio ---
    if len(class_counts) > 1:
        mn, mx = min(class_counts.values()), max(class_counts.values())
        log(f"\n  Imbalance ratio (max/min): {mx/mn:.2f}x", file)
        if mx / mn > 3:
            log(f"  ⚠ HIGH IMBALANCE — consider weighted sampler / loss", file)
        else:
            log(f"  ✔ Roughly balanced", file)

    # --- image sizes ---
    log(f"\n  Image sizes (sampled):", file)
    all_sizes = set()
    for cls, sizes in size_info.items():
        unique = set(sizes)
        all_sizes.update(unique)
        if len(unique) == 1:
            log(f"    {cls:<30} all {list(unique)[0]}", file)
        else:
            ws = [s[0] for s in sizes]; hs = [s[1] for s in sizes]
            log(f"    {cls:<30} w={min(ws)}-{max(ws)}  h={min(hs)}-{max(hs)}  ({len(unique)} unique sizes)", file)

    if len(all_sizes) == 1:
        log(f"\n  ✔ All images same size: {list(all_sizes)[0]}", file)
    else:
        log(f"\n  ⚠ Mixed sizes detected — resize needed", file)

    # --- color modes ---
    log(f"\n  Color modes:", file)
    for cls, modes in mode_info.items():
        log(f"    {cls:<30} {modes}", file)
    all_modes = set().union(*mode_info.values()) if mode_info else set()
    if all_modes - {"RGB"}:
        log(f"\n  ⚠ Non-RGB images found: {all_modes} — convert to RGB before training", file)
    else:
        log(f"\n  ✔ All images are RGB", file)

    # --- corrupt files ---
    if corrupt:
        log(f"\n  ⚠ Corrupt / unreadable files ({len(corrupt)}):", file)
        for p, e in corrupt[:10]:
            log(f"    {p}  →  {e}", file)
    else:
        log(f"\n  ✔ No corrupt files detected (in sampled set)", file)


def main():
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        log("DATASET ANALYSIS REPORT", f)
        log(f"Working directory: {Path.cwd()}", f)

        # --- discover structure ---
        log("\n\nDIRECTORY STRUCTURE", f)
        log("-" * 50, f)
        dataset_root = Path("dataset")
        content_root = Path("content")

        for root in [dataset_root, content_root]:
            if root.exists():
                log(f"\n{root}/", f)
                for item in sorted(root.rglob("*"))[:60]:
                    depth = len(item.relative_to(root).parts)
                    prefix = "  " * depth
                    tag = "/" if item.is_dir() else ""
                    log(f"  {prefix}{item.name}{tag}", f)
            else:
                log(f"\n'{root}' does not exist", f)

        # --- find all splits ---
        splits_to_check = [
            (dataset_root / "train", "Original Train"),
            (dataset_root / "test",  "Test"),
            (content_root / "train", "Content Train (post-split)"),
            (content_root / "val",   "Content Val   (post-split)"),
        ]

        for path, name in splits_to_check:
            analyze_split(path, name, f)

        # --- summary recommendation ---
        log("\n\nRECOMMENDATION SUMMARY", f)
        log("=" * 50, f)

        orig_train = dataset_root / "train"
        if orig_train.exists():
            class_dirs = [d for d in orig_train.iterdir() if d.is_dir()]
            counts = {d.name: len(list(d.glob("*"))) for d in class_dirs}
            total  = sum(counts.values())
            mn, mx = min(counts.values()), max(counts.values())
            log(f"  Total train images : {total}", f)
            log(f"  Imbalance ratio    : {mx/mn:.2f}x", f)
            if mx / mn > 5:
                log("  → Use WeightedRandomSampler + class-weighted loss", f)
            elif mx / mn > 2:
                log("  → Use class-weighted loss (mild imbalance)", f)
            else:
                log("  → Balanced dataset — standard training is fine", f)

            avg_per_class = total / len(counts)
            if avg_per_class < 500:
                log("  → Small dataset (<500/class) — use heavy augmentation, small model", f)
            elif avg_per_class < 2000:
                log("  → Medium dataset — EfficientNet-B0/B3 or ViT-Small recommended", f)
            else:
                log("  → Large dataset — ViT-Base or EfficientNet-B4 feasible", f)

        log(f"\nReport saved to: {OUTPUT_FILE}", f)
        log("Done.", f)

if __name__ == "__main__":
    main()

DATASET ANALYSIS REPORT
Working directory: c:\Users\ASUS\Desktop\MiniProject


DIRECTORY STRUCTURE
--------------------------------------------------

dataset/
    test/
      Mild Impairment/
        1 (10).jpg
        1 (11).jpg
        1 (14).jpg
        1 (15).jpg
        1 (2).jpg
        1 (25).jpg
        10 (12).jpg
        10 (2).jpg
        10 (7).jpg
        11 (10).jpg
        11 (14).jpg
        11 (22).jpg
        11 (23).jpg
        11 (24).jpg
        11 (25).jpg
        11 (26).jpg
        11 (28).jpg
        12 (11).jpg
        12 (13).jpg
        12 (16).jpg
        12 (23).jpg
        12 (3).jpg
        13 (14).jpg
        13 (22).jpg
        13 (27).jpg
        13 (3).jpg
        13 (6).jpg
        13 (8).jpg
        14 (15).jpg
        14 (23).jpg
        14 (28).jpg
        14 (7).jpg
        15 (14).jpg
        15 (21).jpg
        15 (24).jpg
        15 (27).jpg
        15 (4).jpg
        16 (14).jpg
        16 (19).jpg
        16 (2).jpg
        16 (22).jpg
   